## 1. Setup & Imports

In [ ]:
# Install dependencies if needed (uncomment for Colab)
# !pip install torch torchtext tqdm scikit-learn matplotlib seaborn

In [ ]:
import sys
import os

# Add project root to path
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from models.lstm import LSTMSentiment, LSTMAttention, count_parameters
from src.data import IMDBDataModule, preprocess_text, tokenize, Vocabulary
from src.engine import Trainer, evaluate
from src.utils import set_seed, get_device, plot_training_curves

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Set seed for reproducibility
set_seed(42)

# Get device
device = get_device()

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

## 2. Understanding the Data

We'll use the **IMDB Movie Reviews** dataset:
- 50,000 movie reviews (25k train, 25k test)
- Binary labels: positive (1) or negative (0)

In [ ]:
# Initialize data module
dm = IMDBDataModule(
    data_dir='../data',
    batch_size=32,
    max_vocab_size=25000,
    max_seq_len=256
)

# Prepare and setup data
dm.prepare_data()
dm.setup()

print(f"\nVocabulary size: {len(dm.vocab)}")
print(f"Train samples: {len(dm.train_dataset)}")
print(f"Test samples: {len(dm.test_dataset)}")

### 2.1 Explore Preprocessing

In [ ]:
# Example of preprocessing
raw_text = """<br><br>This movie was AMAZING!!! I absolutely loved it. 
The acting was superb & the plot kept me on the edge of my seat. 
10/10 would recommend! Check out http://reviews.com for more."""

print("Original:")
print(raw_text)
print("\nAfter preprocessing:")
processed = preprocess_text(raw_text)
print(processed)
print("\nTokenized:")
tokens = tokenize(processed)
print(tokens)

### 2.2 Examine a Batch

In [ ]:
# Get a batch
train_loader = dm.train_dataloader()
batch = next(iter(train_loader))
texts, labels, lengths = batch

print(f"Batch shapes:")
print(f"  Texts: {texts.shape}")
print(f"  Labels: {labels.shape}")
print(f"  Lengths: {lengths[:5]}...")

# Decode first example
first_tokens = dm.vocab.decode(texts[0].tolist())
first_tokens = [t for t in first_tokens if t != '<pad>']  # Remove padding
print(f"\nFirst example:")
print(f"  Text: {' '.join(first_tokens[:30])}...")
print(f"  Label: {'Positive' if labels[0] == 1 else 'Negative'}")

## 3. Model Architecture

Let's understand the LSTM architecture step by step.

In [ ]:
# Create both model variants
model_basic = LSTMSentiment(
    vocab_size=len(dm.vocab),
    embed_dim=100,
    hidden_dim=128,
    num_layers=2,
    dropout=0.5,
    pad_idx=dm.vocab.pad_idx
)

model_attention = LSTMAttention(
    vocab_size=len(dm.vocab),
    embed_dim=100,
    hidden_dim=128,
    num_layers=2,
    dropout=0.5,
    pad_idx=dm.vocab.pad_idx
)

print("Basic LSTM:")
print(f"  Parameters: {count_parameters(model_basic):,}")
print("\nLSTM with Attention:")
print(f"  Parameters: {count_parameters(model_attention):,}")

In [ ]:
# Visualize architecture
print(model_attention)

## 4. Training

In [ ]:
# Use the attention model for training
model = model_attention.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Get loaders
train_loader = dm.train_dataloader()
test_loader = dm.test_dataloader()

print(f"Training on {device}")
print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    patience=3,
    checkpoint_dir='../models'
)

# Train (reduce epochs for demo)
EPOCHS = 3  # Increase for better results

history = trainer.fit(
    train_loader=train_loader,
    val_loader=test_loader,
    epochs=EPOCHS
)

### 4.1 Training Curves

In [ ]:
# Plot training curves
plot_training_curves(history)

## 5. Evaluation

In [ ]:
# Final evaluation
metrics = evaluate(model, test_loader, criterion, device)

print("\n" + "="*50)
print("FINAL TEST RESULTS")
print("="*50)
print(f"  Loss:      {metrics['loss']:.4f}")
print(f"  Accuracy:  {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.2f}%)")
print(f"  F1 Score:  {metrics['f1']:.4f}")
print(f"  Precision: {metrics['precision']:.4f}")
print(f"  Recall:    {metrics['recall']:.4f}")
print("="*50)

## 6. Inference & Attention Visualization

In [ ]:
def predict_with_attention(text: str, model, vocab, device):
    """Predict sentiment and return attention weights."""
    model.eval()
    
    # Preprocess
    tokens = tokenize(preprocess_text(text))
    encoded = vocab.encode(tokens)
    
    # Predict
    with torch.no_grad():
        x = torch.tensor([encoded]).to(device)
        output, attention = model(x, return_attention=True)
        probs = torch.softmax(output, dim=1)
        pred = output.argmax(dim=1).item()
        confidence = probs[0][pred].item()
    
    sentiment = "Positive" if pred == 1 else "Negative"
    attention = attention[0].cpu().numpy()[:len(tokens)]
    
    return sentiment, confidence, tokens, attention

In [ ]:
# Test predictions
test_reviews = [
    "This movie was absolutely fantastic! The acting was brilliant and the story kept me engaged throughout.",
    "Terrible film. Boring plot, bad acting, and a complete waste of time. Do not watch.",
    "It was okay. Some good moments but overall pretty average."
]

for review in test_reviews:
    sentiment, conf, tokens, attn = predict_with_attention(review, model, dm.vocab, device)
    color = '\033[92m' if sentiment == 'Positive' else '\033[91m'
    reset = '\033[0m'
    print(f"\nReview: {review[:60]}...")
    print(f"Prediction: {color}{sentiment}{reset} ({conf:.2%})")

In [ ]:
# Visualize attention for first review
sentiment, conf, tokens, attn = predict_with_attention(test_reviews[0], model, dm.vocab, device)

# Plot attention heatmap
fig, ax = plt.subplots(figsize=(15, 2))

# Limit to first 30 tokens for visibility
n_tokens = min(30, len(tokens))
display_tokens = tokens[:n_tokens]
display_attn = attn[:n_tokens]

ax.imshow([display_attn], cmap='Reds', aspect='auto')
ax.set_xticks(range(n_tokens))
ax.set_xticklabels(display_tokens, rotation=45, ha='right')
ax.set_yticks([])
ax.set_title(f"Attention Weights - Prediction: {sentiment} ({conf:.2%})")

plt.tight_layout()
plt.show()

## 7. Save Model

In [ ]:
# Save final model
save_path = '../models/lstm_sentiment_final.pth'

torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_size': len(dm.vocab),
    'history': history
}, save_path)

print(f"✓ Model saved to {save_path}")

---

## Summary

In this notebook we:

1. **Loaded and preprocessed** the IMDB dataset
2. **Built a vocabulary** and encoded text to numerical sequences
3. **Implemented LSTM models** (basic and with attention)
4. **Trained** the model with early stopping
5. **Evaluated** on test data
6. **Visualized attention** to understand model decisions

### Next Steps

- Try **pretrained embeddings** (GloVe, Word2Vec)
- Experiment with **GRU** instead of LSTM
- Add **bidirectional** attention
- Try **Transformer** models (BERT, RoBERTa) for comparison